# Chapitre 5 - Exercices 5.16 à 5.19

Régression ridge et noyaux, fonctions propres d'un noyau estimées à partir de la matrice
de Gram, moments nuls des ondelettes, transformée de Haar rapide.

In [1]:
import time
import numpy as np
import matplotlib.pyplot as plt
from numpy.polynomial.hermite import hermval
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl.ch05_splines import haar_transform, inverse_haar_transform, haar_matrix
set_style()

## Exercice 5.16 : Ridge et noyau

**Énoncé.** On considère la régression ridge (5.53) avec $M \ge N$ fonctions de base et
un noyau $K(x, y) = \sum_{m=1}^{M}h_m(x)h_m(y) = h(x)^Th(y)$.

(a) Établir (5.62) : $h(x) = \mathbf V\mathbf D_\gamma^{1/2}\phi(x)$. Comment calculer
    $\mathbf V$ et $\mathbf D_\gamma$ à partir de $K$ ? En déduire que le problème
    (5.63), $\min_\beta\sum_i(y_i - \sum_m\beta_mh_m(x_i))^2 + \lambda\sum_m\beta_m^2$,
    équivaut à (5.53), $\min_c\sum_i(y_i - \sum_jc_j\phi_j(x_i))^2 + \lambda\sum_jc_j^2/\gamma_j$.
(b) Montrer (5.75) : $\hat{\mathbf f} = \mathbf H\hat\beta = \mathbf K(\mathbf K +
    \lambda\mathbf I)^{-1}\mathbf y$, où $\mathbf H = (h_m(x_i))$ est $N \times M$ et
    $\mathbf K = \mathbf H\mathbf H^T$.
(c) Montrer (5.76) : $\hat f(x) = h(x)^T\hat\beta = \sum_iK(x, x_i)\hat\alpha_i$ avec
    $\hat\alpha = (\mathbf K + \lambda\mathbf I)^{-1}\mathbf y$.
(d) Que faire si $M < N$ ?

**(a) Lien entre $h$ et la décomposition spectrale.** Les fonctions propres $\phi_i$ de
(5.45) sont orthonormées pour un produit scalaire $\langle u, v\rangle = \int u(x)v(x)\,
d\mu(x)$ (une mesure $\mu$ sur l'espace des $x$, par exemple la loi des données). Posons
$\mathbf G = \int h(x)h(x)^T\,d\mu(x)$, matrice $M \times M$ symétrique semi-définie
positive, et diagonalisons-la : $\mathbf G = \mathbf V\mathbf D_\gamma\mathbf V^T$ avec
$\mathbf V$ orthogonale. Supposons $\gamma_j > 0$ et définissons
$\phi(x) = \mathbf D_\gamma^{-1/2}\mathbf V^Th(x)$, c'est-à-dire
$h(x) = \mathbf V\mathbf D_\gamma^{1/2}\phi(x)$ (5.62). Alors

- $\int\phi\phi^T\,d\mu = \mathbf D_\gamma^{-1/2}\mathbf V^T\mathbf G\mathbf V\mathbf
  D_\gamma^{-1/2} = \mathbf I$ : les $\phi_j$ sont orthonormées ;
- $\sum_j\gamma_j\phi_j(x)\phi_j(y) = \phi(x)^T\mathbf D_\gamma\phi(y) = h(x)^T\mathbf
  V\mathbf D_\gamma^{-1/2}\mathbf D_\gamma\mathbf D_\gamma^{-1/2}\mathbf V^Th(y) =
  h(x)^Th(y) = K(x, y)$.

C'est donc bien la décomposition (5.45) de $K$.

**Calcul à partir de $K$.** On ne connaît que $K$ : on prend pour $\mu$ la mesure
empirique des $x_i$. Alors $\mathbf G = \frac1N\mathbf H^T\mathbf H$, dont les valeurs
propres non nulles sont celles de $\frac1N\mathbf H\mathbf H^T = \frac1N\mathbf K$ (si
$\mathbf H^T\mathbf H v = \gamma v$, alors $\mathbf H\mathbf H^T(\mathbf Hv) = \gamma\,\mathbf Hv$).
Si $\frac1N\mathbf K = \mathbf U\mathbf D_\gamma\mathbf U^T$, les vecteurs propres
correspondants de $\mathbf G$ sont $\mathbf V = \frac{1}{\sqrt N}\mathbf H^T\mathbf
U\mathbf D_\gamma^{-1/2}$ ; quant aux valeurs $\phi(x_i)$, ce sont les lignes de
$\sqrt N\,\mathbf U$ (voir aussi l'exercice 5.17).

**Équivalence de (5.63) et (5.53).** Avec $c = \mathbf D_\gamma^{1/2}\mathbf V^T\beta$
(bijection, car $\mathbf V$ orthogonale et $\gamma_j > 0$) :
$\sum_m\beta_mh_m(x) = \beta^T\mathbf V\mathbf D_\gamma^{1/2}\phi(x) = c^T\phi(x)$ et
$\|\beta\|^2 = \|\mathbf V\mathbf D_\gamma^{-1/2}c\|^2 = c^T\mathbf D_\gamma^{-1}c =
\sum_jc_j^2/\gamma_j$. Les deux critères sont égaux.

**(b)** La solution ridge est $\hat\beta = (\mathbf H^T\mathbf H + \lambda\mathbf
I_M)^{-1}\mathbf H^T\mathbf y$. L'identité « de passage »

$$
(\mathbf H^T\mathbf H + \lambda\mathbf I_M)^{-1}\mathbf H^T = \mathbf H^T(\mathbf
H\mathbf H^T + \lambda\mathbf I_N)^{-1}
$$

se vérifie en partant de $\mathbf H^T(\mathbf H\mathbf H^T + \lambda\mathbf I) =
(\mathbf H^T\mathbf H + \lambda\mathbf I)\mathbf H^T$ et en multipliant à gauche par
$(\mathbf H^T\mathbf H + \lambda\mathbf I)^{-1}$ et à droite par $(\mathbf H\mathbf H^T +
\lambda\mathbf I)^{-1}$ (inversibles pour $\lambda > 0$, car symétriques définies
positives). Donc $\hat{\mathbf f} = \mathbf H\hat\beta = \mathbf H\mathbf H^T(\mathbf
H\mathbf H^T + \lambda\mathbf I)^{-1}\mathbf y = \mathbf K(\mathbf K + \lambda\mathbf
I)^{-1}\mathbf y$.

**(c)** De même $\hat f(x) = h(x)^T\hat\beta = h(x)^T\mathbf H^T(\mathbf K +
\lambda\mathbf I)^{-1}\mathbf y = \sum_ih(x)^Th(x_i)\,\hat\alpha_i = \sum_iK(x,
x_i)\hat\alpha_i$, avec $\hat\alpha = (\mathbf K + \lambda\mathbf I)^{-1}\mathbf y$.